# Image Sampling

Copies a small representative sample of panoramic photos from the HDD to a local folder.
Used for visual inspection and testing — does not feed into the main analysis pipeline.

**Sampling strategy:** every 100th photo from the CSV (indices 0, 100, 200, ...),
which spreads the selection evenly across the full dataset rather than grabbing consecutive photos.

**Input files:**
- `D:\rotterdam_aiis_2025\vault-production\vault_v1\Recording.csv` — full photo index (706,743 rows, tab-delimited)
- `D:\rotterdam_aiis_2025\vault-production\vault_v1\JPEG\` — panoramic photos on HDD

**Output files:**
- `data/images/` — 100 sampled panoramas copied locally for inspection

**Does not depend on:** notebooks 00, 02, 03, 04, or 05.

In [ ]:
import pandas as pd
import shutil
import os

# Source paths on the HDD
CSV_PATH = r"D:\rotterdam_aiis_2025\vault-production\vault_v1\Recording.csv"
JPEG_DIR = r"D:\rotterdam_aiis_2025\vault-production\vault_v1\JPEG"

# Local destination folder (created automatically if it doesn't exist)
OUTPUT_DIR = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second year\Afstuderen\Project\data\images"

# Sampling parameters
N_IMAGES  = 100   # total number of images to copy
STEP_SIZE = 100   # skip this many images between each selected one

In [ ]:
# Read the CSV with tab delimiter (as established earlier)
df = pd.read_csv(CSV_PATH, sep="\t")
df.columns = df.columns.str.strip()

print(f"Total photos in CSV: {len(df):,}")
df.head()

In [ ]:
# Select every 100th row up to 100 images
# iloc[::STEP_SIZE] takes rows at indices 0, 100, 200, 300, ...
# [:N_IMAGES] then limits to the first 100 of those
sample = df.iloc[::STEP_SIZE][:N_IMAGES].copy()

print(f"Selected {len(sample)} images")
print(f"First index: {sample.index[0]}, Last index: {sample.index[-1]}")
sample[["Filename", "X/Long", "Y/Lat"]].head(10)

In [ ]:
# Create the output directory if it doesn't exist yet
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output folder: {OUTPUT_DIR}")

copied  = 0  # count of successfully copied files
missing = 0  # count of files not found on HDD

for _, row in sample.iterrows():
    # Build source and destination file paths
    src  = os.path.join(JPEG_DIR, row["Filename"] + ".jpeg")
    dst  = os.path.join(OUTPUT_DIR, row["Filename"] + ".jpeg")

    if os.path.exists(src):
        shutil.copy2(src, dst)  # copy2 preserves file metadata
        copied += 1
    else:
        print(f"  Not found: {src}")
        missing += 1

print(f"\nDone! Copied: {copied}  |  Missing: {missing}")
print(f"Images saved to: {OUTPUT_DIR}")